# SCADA / ICS Anomaly Detection System
> **AI-Powered Cyberattack Detection on Industrial Control Systems using Random Forest and Splunk Cloud SIEM Integration**

This notebook implements an end-to-end cyberattack detection pipeline on the **SWaT (Secure Water Treatment)** dataset. It trains a **Random Forest Classifier** to detect cyber-physical anomalies across 51 industrial sensors and forwards real-time alerts with confidence and severity ratings to **Splunk SIEM** via HTTP Event Collector (HEC).

In [1]:
# ── Install dependencies ──
!pip install scikit-learn pandas numpy matplotlib seaborn requests -q

# ── Import libraries ──
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

print("✅ All libraries loaded successfully!")

✅ All libraries loaded successfully!


In [2]:
# ── Mount Google Drive ──
from google.colab import drive
drive.mount('/content/drive')

print("✅ Google Drive mounted!")

✅ Google Drive mounted!


In [3]:
# ── Set dataset paths ──
normal_path = '/content/drive/MyDrive/ScadaML/Datasets/normal.csv'
attack_path = '/content/drive/MyDrive/ScadaML/Datasets/attack.csv'

print("✅ Paths set!")

✅ Paths set!


In [4]:
# ── Load datasets ──
normal_df = pd.read_csv(normal_path)
attack_df  = pd.read_csv(attack_path)

print("✅ Normal data shape:", normal_df.shape)
print("✅ Attack data shape:", attack_df.shape)

In [5]:
# ── Strip whitespace from column names ──
normal_df.columns = normal_df.columns.str.strip()
attack_df.columns = attack_df.columns.str.strip()

print("✅ Columns cleaned!")
print(normal_df.columns.tolist())

In [6]:
# ── Check for missing values ──
print("Missing values in Normal:", normal_df.isnull().sum().sum())
print("Missing values in Attack:", attack_df.isnull().sum().sum())

# ── Extract feature columns (51 sensors) ──
feature_cols = [col for col in normal_df.columns if col not in ['Timestamp', 'Normal/Attack']]
print(f"\n✅ Total sensor features: {len(feature_cols)}")

In [7]:
# ── Prepare labeled dataset for Random Forest ──
# Handle missing values by imputing feature means
normal_sample = normal_df[feature_cols].fillna(normal_df[feature_cols].mean()).sample(200000, random_state=42)
attack_sample = attack_df[feature_cols].fillna(attack_df[feature_cols].mean())

normal_sample['label'] = 0  # 0 = Normal
attack_sample['label'] = 1  # 1 = Attack

full_df = pd.concat([normal_sample, attack_sample], ignore_index=True)
X = full_df.drop('label', axis=1)
y = full_df['label']

# ── Stratified 80/20 Train-Test Split ──
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ── Feature Normalization ──
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print("✅ Dataset prepared!")
print("Train size:", len(X_train_scaled))
print("Test size:", len(X_test_scaled))

✅ Dataset prepared!
Train size: 203700
Test size: 50925


In [8]:
# ── Train Random Forest Classifier ──
print("⏳ Training Random Forest Classifier...")

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train_scaled, y_train)

print("✅ Random Forest model trained successfully!")

⏳ Training Random Forest Classifier...
✅ Random Forest model trained successfully!


In [9]:
# ── Evaluate Random Forest Model ──
y_pred = rf_model.predict(X_test_scaled)

print("=== RANDOM FOREST MODEL PERFORMANCE ===\n")
print(classification_report(y_test, y_pred, target_names=['Normal', 'Attack'], digits=4))

# ── Confusion Matrix Heatmap ──
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=['Normal', 'Attack'],
            yticklabels=['Normal', 'Attack'])
plt.title('Confusion Matrix — Random Forest')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

=== RANDOM FOREST MODEL PERFORMANCE ===

              precision    recall  f1-score   support

      Normal     1.0000    1.0000    1.0000     40001
      Attack     1.0000    1.0000    1.0000     10924

    accuracy                         1.0000     50925
   macro avg     1.0000    1.0000    1.0000     50925
weighted avg     1.0000    1.0000    1.0000     50925



In [10]:
# ── Feature Importance Analysis ──
importances = rf_model.feature_importances_
feat_importances = pd.Series(importances, index=feature_cols).sort_values(ascending=False)

print("Top 10 Most Critical Sensors Detected by Random Forest:")
print(feat_importances.head(10))

# ── Plot Top 15 Sensors ──
plt.figure(figsize=(10, 6))
sns.barplot(x=feat_importances.head(15).values, y=feat_importances.head(15).index, palette='viridis')
plt.title("Top 15 Most Critical SCADA Sensors for Cyberattack Detection")
plt.xlabel("Gini Importance")
plt.ylabel("Sensor Tag")
plt.tight_layout()
plt.show()

In [11]:
# ── Splunk Cloud HTTP Event Collector (HEC) Config ──
SPLUNK_URL   = "https://prd-p-7w3nf.splunkcloud.com:8088/services/collector/event"
SPLUNK_TOKEN = "<Your Token here>"

# ── Test HEC Connection ──
import requests, json, urllib3
urllib3.disable_warnings()

test_payload = {
    "event": "SCADA IDS Test Alert",
    "sourcetype": "scada_ids_alert"
}
headers = {
    "Authorization": f"Splunk {SPLUNK_TOKEN}",
    "Content-Type" : "application/json"
}

r = requests.post(SPLUNK_URL, data=json.dumps(test_payload), headers=headers, verify=False)
print("Status:", r.status_code)
print("Response:", r.text)

Status: 200
Response: {"text":"Success","code":0}


In [12]:
# ── Alert Dispatcher to Splunk SIEM ──
from datetime import datetime

def send_to_splunk(sensor_data, prediction, confidence):
    severity = "HIGH" if confidence > 0.85 else "MEDIUM"

    payload = {
        "time"      : datetime.utcnow().timestamp(),
        "sourcetype": "scada_ids_alert",
        "index"     : "main",
        "event"     : {
            "alert_type" : "CYBERATTACK DETECTED",
            "severity"   : severity,
            "confidence" : round(float(confidence), 4),
            "sensors"    : {
                "LIT101" : round(float(sensor_data['LIT101']), 3),
                "FIT101" : round(float(sensor_data['FIT101']), 3),
                "AIT202" : round(float(sensor_data['AIT202']), 3),
                "DPIT301": round(float(sensor_data['DPIT301']), 3),
            },
            "timestamp"  : datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S"),
            "source"     : "SWaT_ICS_Anomaly_Detector"
        }
    }

    headers = {
        "Authorization": f"Splunk {SPLUNK_TOKEN}",
        "Content-Type" : "application/json"
    }

    r = requests.post(SPLUNK_URL, data=json.dumps(payload), headers=headers, verify=False)
    return r.status_code

# ── Run predictions on attack data ──
X_attack = scaler.transform(attack_df[feature_cols].fillna(attack_df[feature_cols].mean()))
predictions   = rf_model.predict(X_attack)
probabilities = rf_model.predict_proba(X_attack)[:, 1]

# ── Send first 20 attack alerts to Splunk ──
alerts_sent = 0
for i in range(len(predictions)):
    if predictions[i] == 1:
        status = send_to_splunk(
            sensor_data = attack_df[feature_cols].iloc[i],
            prediction  = predictions[i],
            confidence  = probabilities[i]
        )
        alerts_sent += 1
        if alerts_sent <= 5:
            print(f"Alert {alerts_sent} → Status: {status}")
        if alerts_sent == 20:
            break

print(f"\n✅ {alerts_sent} alerts sent to Splunk!")

Alert 1 → Status: 200
Alert 2 → Status: 200
Alert 3 → Status: 200
Alert 4 → Status: 200
Alert 5 → Status: 200

✅ 20 alerts sent to Splunk!
